In [1]:
import os

import torch
import numpy as np
import plotly.express as px
from torch import nn, Tensor
from matplotlib import pyplot as plt
from torchvision.transforms import v2

from src import configs as cfg
from src import dataset, models

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
def load_chkpt(chkpt_pth: str) -> torch.nn.Module:
    chkpt = torch.load(chkpt_pth, weights_only=False)
    model_cfg = cfg.ModelConfig(**chkpt["model_cfg"])
    model = models.mk_model_from_cfg(model_cfg)
    model.load_state_dict(chkpt["model"])
    model = model.eval()
    return model

In [46]:
train_cfg = cfg.TrainingConfig(batch_size=2)
loaders = dataset.mk_segmentation_data_loaders(train_cfg)
x, y_true = next(iter(loaders["train"]))
batch = dataset.preprocess_batch({"x": x, "y_true": y_true})

CHKPT_DIRECTORY = "checkpoints/unet/absurd-glitter-526/"
N_CHKPT_TO_PLT = -1
chkpt_filenames = os.listdir(CHKPT_DIRECTORY)[:N_CHKPT_TO_PLT]
segs_buffer = torch.empty(
    len(chkpt_filenames), train_cfg.batch_size, 256, 256,
    dtype=torch.uint8,
    device=cfg.DEVICE,
)

with torch.no_grad():
    for chkpt_idx, chkpt_filename in enumerate(chkpt_filenames):
        chkpt_pth = os.path.join(CHKPT_DIRECTORY, chkpt_filename)
        model = load_chkpt(chkpt_pth)
        segs_buffer[chkpt_idx] = (
            model(batch)["y_pred"]
            .argmax(dim=1)
            .to(dtype=torch.uint8)
        )

sampling method: shuffle
train_dl_kwargs: {'shuffle': True, 'batch_size': 2}


In [49]:
N_SAMPLES_TO_SHOW = 10
COLOR_MAP_NAME = "rainbow"
segs_buffer_np = segs_buffer.cpu().numpy()
colored_seg_buffer = plt.cm.rainbow(segs_buffer_np)
colored_x = plt.cm.gray(batch["x"].cpu().numpy()[None, :, 0])
seg_mask = (segs_buffer_np == 0)[..., None]
test_img_buffer = np.where(seg_mask, colored_x, colored_seg_buffer)
test_img_buffer = test_img_buffer[..., :3] # Removed Remove alpha channel
print(test_img_buffer.shape)
# test_img = test_img_buffer.reshape(-1, *test_img_buffer.shape[2:])[0]
px.imshow(
    test_img_buffer, #.reshape(-1, *test_img_buffer.shape[2:]),
    animation_frame=0,
    facet_col=1,
    facet_col_wrap=train_cfg.batch_size,
)

(30, 2, 256, 256, 3)
